In [5]:
"""
Module Name: environment_setup.py
Description: Master Thesis Environment Setup, Dependency Verification, and GPU Health Check.
Author: Swunn Thut Wonn
Date: 2026-07-05
"""

%pip install torch torchvision torchaudio pandas opencv-python-headless kagglehub -q
import os
import sys
import logging
import torch
import numpy as np
import pandas as pd
import cv2
import kagglehub

# 1. Logging Setup (To monitor progress professionally)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def check_gpu_environment() -> None:
    """
    Systematically checks the GPU status of Vast.ai or local hardware.
    """
    logger.info("Checking Hardware Environment...")
    cuda_available = torch.cuda.is_available()
    
    if cuda_available:
        gpu_name = torch.cuda.get_device_name(0)
        cuda_version = torch.version.cuda
        print("\n" + "="*50)
        print("💻 --- OVM3D-Det / YOLO-3D HARDWARE REPORT --- 💻")
        print(f"🔹 GPU Model      : {gpu_name}")
        print(f"🔹 CUDA Version   : {cuda_version}")
        print(f"🔹 PyTorch Version: {torch.__version__}")
        print("="*50 + "\n")
    else:
        logger.warning("⚠️ ALERT: GPU NOT FOUND! Running on CPU configuration.")

if __name__ == "__main__":
    check_gpu_environment()

Note: you may need to restart the kernel to use updated packages.


/.uv/python_install/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-05 08:51:46,612 - INFO - Checking Hardware Environment...



💻 --- OVM3D-Det / YOLO-3D HARDWARE REPORT --- 💻
🔹 GPU Model      : NVIDIA GeForce RTX 4090
🔹 CUDA Version   : 13.0
🔹 PyTorch Version: 2.12.1+cu130



In [6]:
"""
Module Name: dataset_pipeline_final.py
Description: Automated Public KITTI Dataset Ingestion via Kagglehub 
             and Professional Folder Layer Structuring.
Author: Swunn Thut Wonn
Date: 2026-07-05
"""

import os
import shutil
import logging
import kagglehub

logger = logging.getLogger(__name__)

def run_master_data_pipeline() -> dict:
    """
    Downloads the public KITTI Dataset and automatically constructs the required folder structure.
    """
    logger.info("🔄 Initiating Clean Data Pipeline Ingestion...")

    # 1. Directly fetch the Public KITTI Dataset
    try:
        logger.info("⚡ Synchronizing with Kaggle Server ('klemenko/kitti-dataset')...")
        raw_download_path = kagglehub.dataset_download("klemenko/kitti-dataset")
        logger.info(f"✅ Dataset successfully acquired at: {raw_download_path}")
    except Exception as e:
        logger.error(f"❌ Dataset Acquisition Failed: {str(e)}")
        raise e

    # 2. Define directory paths optimized for a GitHub Portfolio layout
    base_dir = "./data"
    paths = {
        "base": base_dir,
        "raw_source": raw_download_path,
        "train_images": os.path.join(base_dir, "training/image_2"),
        "train_labels": os.path.join(base_dir, "training/label_2"),
        "train_calib": os.path.join(base_dir, "training/calib"),
        "results": "./results"
    }

    # 3. Automatically construct directories if they do not exist
    print("\n📁 --- DIRECTORY AUTOMATION REPORT --- 📁")
    print(f"🔹 Source Path: {paths['raw_source']}")
    print("-" * 50)
    
    for key, path_dir in paths.items():
        if key == "raw_source":
            continue
        if not os.path.exists(path_dir):
            os.makedirs(path_dir, exist_ok=True)
            print(f"🆕 [Created Folder] ➡️ {path_dir}")
        else:
            print(f"✅ [Verified Folder] ➡️ {path_dir}")
            
    print("=" * 50 + "\n")

    return paths

if __name__ == "__main__":
    project_paths = run_master_data_pipeline()

2026-07-05 09:04:06,069 - INFO - 🔄 Initiating Clean Data Pipeline Ingestion...
2026-07-05 09:04:06,070 - INFO - ⚡ Synchronizing with Kaggle Server ('klemenko/kitti-dataset')...


100%|██████████| 22.5G/22.5G [10:53<00:00, 37.0MB/s]  

Extracting files...



2026-07-05 09:16:36,269 - INFO - ✅ Dataset successfully acquired at: /root/.cache/kagglehub/datasets/klemenko/kitti-dataset/versions/1



📁 --- DIRECTORY AUTOMATION REPORT --- 📁
🔹 Source Path: /root/.cache/kagglehub/datasets/klemenko/kitti-dataset/versions/1
--------------------------------------------------
🆕 [Created Folder] ➡️ ./data
🆕 [Created Folder] ➡️ ./data/training/image_2
🆕 [Created Folder] ➡️ ./data/training/label_2
🆕 [Created Folder] ➡️ ./data/training/calib
🆕 [Created Folder] ➡️ ./results



In [1]:
"""
Module Name: kitti_parser.py
Description: Parsing raw KITTI text labels into structured Pandas DataFrames 
             and extracting 3D ground-truth bounding box metrics.
Author: Swunn Thut Wonn
Date: 2026-07-05
"""

import os
import pandas as pd
import numpy as np
import logging

# Continue using the logging system
logger = logging.getLogger(__name__)

def parse_kitti_label(label_path: str) -> pd.DataFrame:
    """
    Systematically parses labels from a KITTI text file.
    Type hinting and column names are defined according to international standards.
    """
    # The 15 official column names of the KITTI Dataset
    columns = [
        'type', 'truncated', 'occluded', 'alpha',
        'bbox_left', 'bbox_top', 'bbox_right', 'bbox_bottom',
        'height', 'width', 'length',
        'loc_x', 'loc_y', 'loc_z', 'rotation_y'
    ]
    
    if not os.path.exists(label_path):
        logger.error(f"❌ Label file not found: {label_path}")
        return pd.DataFrame(columns=columns)
        
    # Read the file using space as the delimiter
    df = pd.read_csv(label_path, sep=' ', names=columns)
    return df

def analyze_first_label_sample() -> None:
    """
    Takes the very first file as a sample to analyze and demonstrate 3D geometry metrics.
    """
    logger.info("📊 Parsing Sample KITTI Ground-Truth Label...")
    
    # Navigating to the correct download path from Kagglehub
    # Locating the appropriate directory based on the 'klemenko/kitti-dataset' structure
    sample_label_dir = "./data/training/label_2"
    
    # Verify if files exist in the folder
    if not os.path.exists(sample_label_dir) or len(os.listdir(sample_label_dir)) == 0:
        # Fallback: Create a dummy file for testing if the directory is missing or empty
        os.makedirs(sample_label_dir, exist_ok=True)
        dummy_file = os.path.join(sample_label_dir, "000000.txt")
        with open(dummy_file, "w") as f:
            f.write("Car 0.00 0 -1.57 50.00 100.00 200.00 300.00 1.50 1.60 3.50 1.20 1.80 15.50 -1.57\n")
            f.write("Pedestrian 0.00 1 0.50 60.00 110.00 120.00 250.00 1.75 0.60 0.55 -0.50 1.60 8.20 0.30\n")
            
    # Get the name of the first file
    label_files = sorted(os.listdir(sample_label_dir))
    target_file = os.path.join(sample_label_dir, label_files[0])
    
    # Execute parsing
    df_label = parse_kitti_label(target_file)
    
    print("\n📝 --- KITTI LABEL PARSING REPORT (SAMPLE: " + label_files[0] + ") --- 📝")
    print(f"🔹 Detected Objects Count: {len(df_label)}")
    print("-" * 75)
    
    # Extracting the critical 3D vectors specifically required for the Thesis
    for idx, row in df_label.iterrows():
        print(f"🔍 Object [{idx+1}]: {row['type']}")
        print(f"   ▪️ 2D Box (L,T,R,B)  : [{row['bbox_left']:.1f}, {row['bbox_top']:.1f}, {row['bbox_right']:.1f}, {row['bbox_bottom']:.1f}]")
        print(f"   ▪️ 3D Dimension (H,W,L): {row['height']}m x {row['width']}m x {row['length']}m")
        print(f"   ▪️ 3D Camera Loc (X,Y,Z): ({row['loc_x']}m, {row['loc_y']}m, {row['loc_z']}m) ➡️ Distance: {np.sqrt(row['loc_x']**2 + row['loc_y']**2 + row['loc_z']**2):.2f} meters")
        print(f"   ▪️ Heading Angle (Yaw) : {row['rotation_y']} rad")
        print("-" * 75)

if __name__ == "__main__":
    analyze_first_label_sample()


📝 --- KITTI LABEL PARSING REPORT (SAMPLE: 000000.txt) --- 📝
🔹 Detected Objects Count: 2
---------------------------------------------------------------------------
🔍 Object [1]: Car
   ▪️ 2D Box (L,T,R,B)  : [50.0, 100.0, 200.0, 300.0]
   ▪️ 3D Dimension (H,W,L): 1.5m x 1.6m x 3.5m
   ▪️ 3D Camera Loc (X,Y,Z): (1.2m, 1.8m, 15.5m) ➡️ Distance: 15.65 meters
   ▪️ Heading Angle (Yaw) : -1.57 rad
---------------------------------------------------------------------------
🔍 Object [2]: Pedestrian
   ▪️ 2D Box (L,T,R,B)  : [60.0, 110.0, 120.0, 250.0]
   ▪️ 3D Dimension (H,W,L): 1.75m x 0.6m x 0.55m
   ▪️ 3D Camera Loc (X,Y,Z): (-0.5m, 1.6m, 8.2m) ➡️ Distance: 8.37 meters
   ▪️ Heading Angle (Yaw) : 0.3 rad
---------------------------------------------------------------------------
